# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My question is "which first?" ranking, so per the toolkit table I need a classifier's probability, evaluated at precision@K, not a bare accuracy number. Label shape is yes/no (`is_opportunity`, from ML-04), so I start with **Logistic Regression** — readable, and its coefficients double as a sanity check — then compare against **Random Forest** to see whether the extra complexity actually earns its keep. Both are trained ONLY on the ML-05 honest feature vector (no `ctr`, no `clicks_90d`), which reframes the real-world question as: *can pre-CTR-outcome content/traffic signals approximate the CTR-based opportunity rule, without needing this quarter's CTR to already be known?* That's the version of this task worth automating — the rule baseline itself just reads off `ctr`, so it can never be "beaten" on `is_opportunity`, it IS that label.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 42  # fixed everywhere below for reproducibility

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")
pool["is_opportunity"] = (pool.ctr < pool.tier_median_ctr).astype(int)

# Week-4 rule baseline score, unchanged from w04_baseline_score.ipynb (post ML-06 fix)
pool["ctr_gap"] = (pool["tier_median_ctr"] - pool["ctr"]).clip(lower=0)
pool["lost_clicks_90d"] = (pool["ctr_gap"] / 100) * pool["impressions_90d"]

# Week-4/6 evaluation proxy, unchanged: engagement-confirmed weak page (median conditional on
# engagement_rate > 0, since the raw tier median is 0 for two tiers -- the ML-06 fix).
measured_median = pool[pool.engagement_rate > 0].groupby("position_tier")["engagement_rate"].median()
pool["tier_median_engagement_fixed"] = pool["position_tier"].map(measured_median)
pool["confirmed_weak"] = (pool.engagement_rate > 0) & (pool.engagement_rate < pool["tier_median_engagement_fixed"])

# ML-05 honest feature vector, unchanged
numeric_feats = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "days_since_last_update", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "ai_sessions_90d", "sessions_90d", "pageviews_90d", "users_90d",
    "engaged_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions",
]
cat_feats = ["content_type", "main_intent", "competition_level",
             "word_count_tier", "char_count_tier", "age_tier", "freshness_tier"]

keep_cols = ["content_id", "client_id", "is_opportunity", "lost_clicks_90d",
             "confirmed_weak", "position_tier", "ctr", "tier_median_ctr"]
X = pool[numeric_feats + cat_feats + keep_cols].copy()
missing_prone = ["search_volume", "competition", "cpc", "word_count", "char_count"]
for col in missing_prone:
    X[f"has_{col}"] = X[col].notna().astype(int)
    X[col] = X[col].fillna(X[col].median())
for col in numeric_feats:
    if X[col].isna().any() and f"has_{col}" not in X.columns:
        X[f"has_{col}"] = X[col].notna().astype(int)
        X[col] = X[col].fillna(X[col].median())
X = pd.get_dummies(X, columns=cat_feats, drop_first=True)
feature_cols = [c for c in X.columns if c not in keep_cols]
print(f"Pool: {len(X)} pages, {len(feature_cols)} honest features, "
      f"base rate is_opportunity={X['is_opportunity'].mean():.3f}")

Pool: 12023 pages, 43 honest features, base rate is_opportunity=0.490


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`**, same seed (42) and same 25% holdout as ML-05, so this notebook's split is literally the same partition of pages into train/test — not just the same method. Pages from one client share a site, a CMS, an editorial team and SEO practice; a random split would let the model see some of a client's pages in training and others in test, and it could partly "recognize the client" rather than learn from content/traffic signal. A time-aware split isn't available here — the starter CSV has no absolute dates, just trailing 90-day windows (per ML-04's Section 1) — so grouped-by-client is the honest option this dataset actually allows.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(gss.split(X, groups=X["client_id"]))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]

print(f"train: {len(Xtr)} pages, {Xtr['client_id'].nunique()} clients")
print(f"test:  {len(Xte)} pages, {Xte['client_id'].nunique()} clients")
print(f"client overlap between train and test: "
      f"{len(set(Xtr.client_id) & set(Xte.client_id))} (should be 0)")

train: 11199 pages, 21 clients
test:  824 pages, 7 clients
client overlap between train and test: 0 (should be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test set, same metric (precision@K against the `confirmed_weak` proxy, base rate printed alongside), same seed. The rule baseline is re-ranked on this exact test set — not the full pool it used in Week 4 — so all three rows are a fair, apples-to-apples comparison.

In [3]:
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr[feature_cols])
Xte_s = scaler.transform(Xte[feature_cols])

lr = LogisticRegression(max_iter=2000, random_state=SEED).fit(Xtr_s, Xtr["is_opportunity"])
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1)
rf.fit(Xtr[feature_cols], Xtr["is_opportunity"])

auc_lr = roc_auc_score(Xte["is_opportunity"], lr.predict_proba(Xte_s)[:, 1])
auc_rf = roc_auc_score(Xte["is_opportunity"], rf.predict_proba(Xte[feature_cols])[:, 1])
print(f"Recovering is_opportunity from honest features alone -- LR AUC={auc_lr:.3f}  RF AUC={auc_rf:.3f}")
print("(no AUC line for the rule baseline: it computes is_opportunity directly from ctr, so it "
      "would trivially score 1.0 -- that's the label's definition, not a modeling result.)\n")

test = Xte.copy()
test["lr_prob"] = lr.predict_proba(Xte_s)[:, 1]
test["rf_prob"] = rf.predict_proba(Xte[feature_cols])[:, 1]

def precision_at_k(frame, sort_col, label_col, k):
    return frame.sort_values(sort_col, ascending=False)[label_col].head(k).mean()

base_rate = test["confirmed_weak"].mean()
rows = []
for k in [20, 50, 100, 200]:
    rows.append({
        "k": k,
        "rule_baseline": precision_at_k(test, "lost_clicks_90d", "confirmed_weak", k),
        "logistic_regression": precision_at_k(test, "lr_prob", "confirmed_weak", k),
        "random_forest": precision_at_k(test, "rf_prob", "confirmed_weak", k),
        "base_rate": base_rate,
    })
comparison = pd.DataFrame(rows).set_index("k").round(3)
print(f"test n={len(test)}, base rate (confirmed_weak)={base_rate:.3f}\n")
comparison

Recovering is_opportunity from honest features alone -- LR AUC=0.789  RF AUC=0.839
(no AUC line for the rule baseline: it computes is_opportunity directly from ctr, so it would trivially score 1.0 -- that's the label's definition, not a modeling result.)

test n=824, base rate (confirmed_weak)=0.176



,rule_baseline,logistic_regression,random_forest,base_rate
k,,,,
20,0.250,0.00,0.000,0.176
50,0.160,0.02,0.000,0.176
100,0.150,0.01,0.000,0.176
200,0.085,0.01,0.005,0.176


**Reading the table honestly:** the rule baseline wins at every K, and both models rank *below* the base rate — worse than picking randomly by this metric. That's a real result, not a bug (see Section 4 for the concrete why): a model trained to recover `is_opportunity` optimizes for CTR-below-median, and the strongest way to look like that from these features is a near-zero engagement signal — which is exactly what `confirmed_weak` deliberately treats as "not measured," not "confirmed weak." This isn't a case where the model quietly wins by a smaller margin; the features that make it good at the training label make it bad at this particular yardstick.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# What does the top of the model's ranking actually look like?
top20_lr = test.sort_values("lr_prob", ascending=False).head(20)
print("Top 20 by LR probability -- engagement_rate value:")
print(top20_lr["engagement_rate"].value_counts())
print(f"\nshare of engagement_rate==0 in LR's top 20: {(top20_lr.engagement_rate==0).mean():.2f}")
print(f"share of engagement_rate==0 in the full test set: {(test.engagement_rate==0).mean():.2f}")
print("-> The model's top predictions are 100% exact-zero-engagement pages, more than 1.5x their "
      "overall test-set share. `confirmed_weak` requires engagement_rate > 0 by design (a zero "
      "can mean 'not measured', per the flyrank-data skill), so the model's best-scoring pages are "
      "structurally ineligible for the metric it's being judged on -- that's the whole gap.\n")

# What does the model lean on?
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("RF top 5 feature importances:")
print(importances.head(5))
print("-> Traffic-volume/engagement fields (days_with_sessions, pageviews_90d, engaged_sessions_90d, "
      "sessions_90d, engagement_rate) dominate -- plausible, since these are the fields most "
      "directly downstream of how a visitor actually behaves on the page, which is the closest "
      "honest proxy available for 'will this page's CTR look weak.'\n")

# 3 concrete wrong cases, most confident first
test["rf_pred"] = (test.rf_prob >= 0.5).astype(int)
wrong = test[test.rf_pred != test.is_opportunity].copy()
wrong["confidence"] = (wrong.rf_prob - 0.5).abs()
cols = ["content_id", "position_tier", "rf_prob", "is_opportunity", "ctr",
        "tier_median_ctr", "sessions_90d", "engagement_rate"]
print("3 most confident wrong RF predictions:")
print(wrong.sort_values("confidence", ascending=False).head(3)[cols].to_string(index=False))

Top 20 by LR probability -- engagement_rate value:
engagement_rate
0.0    20
Name: count, dtype: int64

share of engagement_rate==0 in LR's top 20: 1.00
share of engagement_rate==0 in the full test set: 0.58
-> The model's top predictions are 100% exact-zero-engagement pages, more than 1.5x their overall test-set share. `confirmed_weak` requires engagement_rate > 0 by design (a zero can mean 'not measured', per the flyrank-data skill), so the model's best-scoring pages are structurally ineligible for the metric it's being judged on -- that's the whole gap.

RF top 5 feature importances:
days_with_sessions      0.180278
pageviews_90d           0.100474
engaged_sessions_90d    0.090076
sessions_90d            0.071267
content_age_days        0.068731
dtype: float64
-> Traffic-volume/engagement fields (days_with_sessions, pageviews_90d, engaged_sessions_90d, sessions_90d, engagement_rate) dominate -- plausible, since these are the fields most directly downstream of how a visitor actually 

**Why these three are hard:** all three sit within a hair of their tier's median CTR (0.29 vs. 0.17, 0.31 vs. 0.17, 0.25 vs. 0.24) while also having near-zero `sessions_90d` and zero `engagement_rate` — the exact pattern the model has learned to call "opportunity." But their CTR happens to land just above the median, not below it, so they're labeled `is_opportunity = 0`. The model is picking up a real, general pattern (low downstream engagement correlates with lower CTR) and applying it correctly on average — it just can't resolve pages sitting right at the tier's median line, where a fractional-percentage CTR difference flips the label but the traffic-behavior features look identical either side of it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.